In [1]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4
import chromadb
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")

In [5]:
vector_store = Chroma(embedding_function= embedding, collection_name="rag_practice")

In [7]:
documents = [
    Document(
    page_content="An employee can have multiple roles in a company, such as manager, developer, and designer.",
    metadata={"source": "tweet", "name": "Alice"},
    id=1,
    ),
        Document(
        page_content="Office day is total 5 days a week, but we can work from home on Fridays.",
        metadata={"source": "news", "name": "Bob"},
        id=2,
    ),
        Document(
        page_content="The office is located at 123 Main Street, Anytown, USA.",
        metadata={"source": "tweet", "name": "Charlie"},
        id=3,
    ),
]

In [12]:
uids = [str(uuid4()) for _ in range(len(documents))]
uids

['81f235c4-1195-40e0-9b33-f091ecc3b7ff',
 '0b22acd8-824d-458b-a3fc-b28fa98c0dd5',
 '6b76b5ed-e449-480b-8e78-d4c028b1228f']

In [13]:
vector_store.add_documents(documents= documents, ids = uids)

['81f235c4-1195-40e0-9b33-f091ecc3b7ff',
 '0b22acd8-824d-458b-a3fc-b28fa98c0dd5',
 '6b76b5ed-e449-480b-8e78-d4c028b1228f']

In [ ]:
data = vector_store.get()
print(data["documents"])

['An employee can have multiple roles in a company, such as manager, developer, and designer.', 'Office day is total 5 days a week, but we can work from home on Fridays.', 'The office is located at 123 Main Street, Anytown, USA.']

In [17]:
updated_documents = [
    Document(
        page_content="I had chocolate chip pancakes and fried eggs for breakfast this morning.",
        metadata={"source": "tweet", "name": "Alice"},
        id=1,
    ), Document(
        page_content="The weather forecast for tomorrow is sunny and warm, with a high of 82 degrees.",
        metadata={"source": "news", "name": "Bob"},
        id=2,
    )
]

In [20]:
vector_store.update_documents(ids = uids[:2], documents = updated_documents)

In [21]:
data = vector_store.get()
print(data["documents"])

['I had chocolate chip pancakes and fried eggs for breakfast this morning.', 'The weather forecast for tomorrow is sunny and warm, with a high of 82 degrees.', 'The office is located at 123 Main Street, Anytown, USA.']


In [22]:
vector_store.delete(ids = uids[-1])

In [23]:
data = vector_store.get()
print(data["documents"])

['I had chocolate chip pancakes and fried eggs for breakfast this morning.', 'The weather forecast for tomorrow is sunny and warm, with a high of 82 degrees.']


In [30]:
query =  "what breakfast for today?"

result = vector_store.similarity_search(query, k=1)

In [31]:
for r in result:
    print(r.page_content)

I had chocolate chip pancakes and fried eggs for breakfast this morning.


In [34]:
result = vector_store.similarity_search_with_score(query, k = 1, )

for res, score in result:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")
    

* [SIM=1.003527] I had chocolate chip pancakes and fried eggs for breakfast this morning. [{'source': 'tweet', 'name': 'Alice'}]


In [37]:
query_embedding = embedding.embed_query(query)
result = vector_store.similarity_search_by_vector(query_embedding, k = 1)

In [38]:
for doc in result:
    print(f"* {doc.page_content} [{doc.metadata}]")

* I had chocolate chip pancakes and fried eggs for breakfast this morning. [{'source': 'tweet', 'name': 'Alice'}]
